# Imports

In [115]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Callable

import kagglehub
from statsmodels.stats.outliers_influence import variance_inflation_factor

RANDOM_SEED = 42
np.random.seed = RANDOM_SEED

# 1. Dataset preparation

## Downloads

In [102]:
path1 = kagglehub.dataset_download("shree0910/online-vs-in-store-shopping-behaviour-dataset")
path2 = kagglehub.dataset_download("vishardmehta/smartphone-battery-health-prediction-dataset")
path3 = kagglehub.dataset_download("ziya07/software-defect-prediction-dataset")
path4 = kagglehub.dataset_download("yasserh/wine-quality-dataset")

In [103]:
shopping_data = pd.read_csv(f"{path1}/{os.listdir(path1)[0]}")
smartphone_data = pd.read_csv(f"{path2}/{os.listdir(path2)[0]}")
smartphone_features = pd.read_csv(f"{path2}/{os.listdir(path2)[1]}")
software_data = pd.read_csv(f"{path3}/{os.listdir(path3)[0]}")
wine_data = pd.read_csv(f"{path4}/{os.listdir(path4)[0]}")

## Preprocessing

### Functions

In [104]:
def cast_int(df, column):
    dict = {}
    i = 0
    for label in df[column].unique():
        dict[label] = i
        i += 1
    
    return df[column].map(dict)

In [105]:
# def vifify(df, column):
#     X = df.drop(column, axis=1)

#     vif_data = pd.DataFrame()
#     vif_data["feature"] = X.columns

#     vif_data["VIF"] = [variance_inflation_factor(X.values, i)
#                             for i in range(len(X.columns))]
    
#     vif_data.sort_values(by="VIF", axis=0, ascending=False, inplace=True)
    
#     return vif_data, X

In [106]:
# def del_coll(df, col, threshold=5):
#     column = col
#     target = df[col]
#     new_shop = df
#     while True:
#         vif, new_shop = vifify(new_shop, column)

#         if vif.iloc[0,1] > threshold:
#             column = vif.iloc[0,0]
        
#         else:
#             print(vif)
#             break

#     new_shop[col] = target
    
#     return new_shop

### Shopping data

In [107]:
shopping_data["shopping_preference"] = shopping_data["shopping_preference"].map({"Store": 0, "Hybrid": 1, "Online": 1})
shopping_data["city_tier"] = shopping_data["city_tier"].map({"Tier 1": 0, "Tier 2": 1, "Tier 3": 2})
shopping_data.drop("gender", axis=1, inplace=True)

In [ ]:
shopping_data

### Smartphone data

In [108]:
smartphone_data = smartphone_features.merge(smartphone_data, on="Device_ID").drop("current_battery_health_percent", axis=1)

In [109]:
smartphone_data["target_action"] = smartphone_data["recommended_action"].map({"Change Phone": 1, "Replace Battery": 1, "Keep Using": 0})
smartphone_data["background_app_usage_level"] = smartphone_data["background_app_usage_level"].map({"Low": 0, "Medium": 1, "High": 2})
smartphone_data["signal_strength_avg"] = smartphone_data["signal_strength_avg"].map({"Poor": 0, "Moderate": 1, "Good": 2})

smartphone_data.drop(["recommended_action", "Device_ID"], axis=1, inplace=True)

### Software data

In [110]:
software_data.describe()

,LOC,CYCLO,LENGTH,VOLUME,DIFFICULTY,INT_FAN_IN,INT_FAN_OUT,NUM_OPERATORS,NUM_OPERANDS,BRANCH_COUNT,DEFECT_LABEL
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.520711,0.493304,0.509783,0.505971,0.502349,0.503556,0.510667,0.493988,0.512625,0.514643,0.326000
std,0.289402,0.301158,0.289705,0.291389,0.284572,0.315975,0.321400,0.295131,0.276572,0.314337,0.468982
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.278720,0.217391,0.255017,0.252619,0.262725,0.222222,0.222222,0.231084,0.280738,0.214286,0.000000
50%,0.531834,0.478261,0.513841,0.522552,0.511328,0.555556,0.555556,0.494888,0.517418,0.571429,0.000000
75%,0.767474,0.739130,0.761678,0.759446,0.750681,0.777778,0.777778,0.756646,0.742828,0.785714,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### Wine data

In [111]:
wine_data = wine_data.drop('Id', axis=1)

In [112]:
wine_data["quality"].unique()

array([5, 6, 7, 4, 8, 3])

In [113]:
def cast_wine(df, column):
    dict = {}
    for label in df[column].unique():
        if label > 5:
            dict[label] = 1
        else:
            dict[label] = 0
    
    return df[column].map(dict)

wine_data["quality"] = cast_wine(wine_data, "quality")

### Assumed binary values for each 
- Shopping: 0 -- stationary,   1 -- online
- Smartphones: 0 -- no action,  1 -- replace battery or phone
- Software: 0 -- clean,  1 -- defective
- Wine: 0 -- poor quality,  1 -- good quality

## Missing data generation

In [ ]:
def remove_labels(df: pd.DataFrame, target: str, scheme: str) -> pd.DataFrame:
    """
    Remove labels in the dataset based on the chosen scheme. A new column named `target`_`scheme` is added to the original DataFrame.

    Schemes:
        MCAR: Missing Completely At Random. P(S=1|X,Y)=P(S=1)=c.
              Labels are removed uniformly at random with probability c.

        MAR1: Missing At Random 1. P(S=1|X,Y)=P(S=1|X).
              Missingness depends only on a single explanatory variable.

        MAR2: Missing At Random 2. P(S=1|X,Y)=P(S=1|X).
              Firstly, all feature columns are normalized to interval [0, 1]. Then, for each row, the mean of values is calculated.
              The resulting series is categorized into 10 bins, of which an arbitrarily chosen bin (in this implementation, bin 3)
              is used to make labels more likely to be missing.

        MNAR: Missing Not At Random. P(S=1|X,Y).
              Analogous to MAR2 with one exception: if an observation belongs to class 1, the mean corresponding to it is transformed
              using formula x -> 1 - x.

    Parameters:
        df (pandas.DataFrame): A Dataframe containing a column in which the labels are to be removed.
        target (str): The name of the column containing labels.
        scheme (str): Scheme for generating missing labels: `"mcar"`, `"mar1"`, `"mar2"` or `"mnar"`.

    Returns:
        df (pandas.DataFrame): The original DataFrame with a new column containing removed labels (missing labels are set to -1).
    """

    generator = np.random.default_rng(seed=RANDOM_SEED)

    if scheme == "mcar":
        missing_prob = 0.10
        is_missing = generator.binomial(1, missing_prob, size=len(df))

    elif scheme == "mar1":
        feature_cols = df.columns.drop(list(df.filter(regex=target)))
        if len(feature_cols) == 0:
            raise ValueError("MAR1 requires at least one explanatory variable column.")

        mar_feature = df[feature_cols].nunique().idxmax()
        x = pd.to_numeric(df[mar_feature], errors="coerce")
        x = x.fillna(x.mean())

        denom = x.max() - x.min()
        if denom == 0 or np.isnan(denom):
            normalized_x = pd.Series(0.5, index=df.index)
        else:
            normalized_x = (x.max() - x) / denom

        deciles = pd.qcut(normalized_x, 10, labels=False, duplicates="drop")
        chosen_bin = min(3, int(deciles.max()))
        missing_prob = deciles.map(lambda d: 0.9 if d == chosen_bin else 0.05)
        is_missing = generator.binomial(1, missing_prob)

    elif scheme == "mar2":
        normalized_value_means = df[df.columns.drop(list(df.filter(regex=target)))].apply(
            lambda x: (x.max() - x) / (x.max() - x.min()), axis=0
        ).mean(axis=1)
        deciles = pd.qcut(normalized_value_means, 10, labels=False)
        is_missing = generator.binomial(1, deciles.map(lambda x: 0.9 if x == 3 else 0.05))

    elif scheme == "mnar":
        normalized_value_means = df[df.columns.drop(list(df.filter(regex=target)))].apply(
            lambda x: (x.max() - x) / (x.max() - x.min()), axis=0
        ).mean(axis=1)
        normalized_value_means[df[target] == 1] = 1 - normalized_value_means[df[target] == 1]
        deciles = pd.qcut(normalized_value_means, 10, labels=False)
        is_missing = generator.binomial(1, deciles.map(lambda x: 0.9 if x == 3 else 0.05))

    else:
        raise ValueError("Argument scheme accepts only one of following values: 'mcar', 'mar1', 'mar2' or 'mnar'.")

    is_missing = np.asarray(is_missing, dtype=bool)
    print(f"Percentage of labels removed: {is_missing.mean()*100:.2f}%")

    df[target + "_" + scheme] = df[target]
    df.loc[is_missing, target + "_" + scheme] = -1
    return df

### MCAR

In [ ]:
shopping_data = remove_labels(shopping_data, "shopping_preference", "mcar")
smartphone_data = remove_labels(smartphone_data, "target_action", "mcar")
blood_data = remove_labels(blood_data, "Class", "mcar")
wine_data = remove_labels(wine_data, "quality", "mcar")

### MAR 1

In [ ]:
shopping_data = remove_labels(shopping_data, "shopping_preference", "mar1")
smartphone_data = remove_labels(smartphone_data, "target_action", "mar1")
blood_data = remove_labels(blood_data, "Class", "mar1")
wine_data = remove_labels(wine_data, "quality", "mar1")

### MAR 2

In [ ]:
shopping_data = remove_labels(shopping_data, "shopping_preference", "mar2")
smartphone_data = remove_labels(smartphone_data, "target_action", "mar2")
blood_data = remove_labels(blood_data, "Class", "mar2")
wine_data = remove_labels(wine_data, "quality", "mar2")

### MNAR

In [ ]:
shopping_data = remove_labels(shopping_data, "shopping_preference", "mnar")
smartphone_data = remove_labels(smartphone_data, "target_action", "mnar")
blood_data = remove_labels(blood_data, "Class", "mnar")
wine_data = remove_labels(wine_data, "quality", "mnar")